# 笔记本 09 — 回测引擎

**第四阶段 · 评估与部署（1 / 2）**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 从零构建一个简单的**滚动前推回测器** |
| 2 | 实现关键绩效指标：**夏普、索提诺、卡尔马、盈利因子、胜率** |
| 3 | 理解策略评估中的**训练 / 验证拆分** |
| 4 | 可视化资金曲线、回撤和滚动夏普 |
| 5 | 与生产级 `core_module_backtester.py` 的指标进行对比 |

### 前置要求
- NB03（风险指标）、NB04–NB08（策略与风控管线）

In [ ]:
# ── 环境设置 ──────────────────────────────────────────────
import sys, pathlib, warnings, math
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
DAYS_PER_YEAR = 365
print("✅ 导入完成  |  项目根目录:", ROOT)

---
## 1 · 回测基础

回测回答的核心问题是：**"如果过去执行这个策略，结果会怎样？"**

### 滚动前推协议

```
          ┌── 训练 (90天) ──┐┌── 验证 (90天) ──┐
时间轴:   ═══════════════════════════════════════▶
          ^                  ^                  ^
        起始               分割点              结束
```

- **训练期：** 优化参数、发现模式
- **验证期：** 在未见数据上评估（禁止偷看！）
- 生产回测器使用 `history_days=180`、`train_days=90`、`validation_days=90`

### 核心指标

| 指标 | 公式 | 含义 |
|------|------|------|
| **夏普比率** | $\frac{\bar{r}}{\sigma_r} \sqrt{N}$ | 风险调整收益（惩罚所有波动） |
| **索提诺比率** | $\frac{\bar{r}}{\sigma_{\text{down}}} \sqrt{N}$ | 类似夏普但仅惩罚下行波动 |
| **卡尔马比率** | $\frac{\text{CAGR}}{|\text{MaxDD}|}$ | 每单位最大回撤的收益 |
| **盈利因子** | $\frac{\sum \text{盈利}}{|\sum \text{亏损}|}$ | 总盈利 / 总亏损 |
| **胜率** | $\frac{\text{盈利天数}}{\text{总天数}}$ | 正收益天数的占比 |

---
## 2 · 合成多资产回测数据

In [ ]:
np.random.seed(42)
N_DAYS = 180
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=N_DAYS, freq="D")

# Generate correlated multi-asset returns
assets = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
n_assets = len(assets)

# Market factor + idiosyncratic
market_factor = np.random.normal(0.001, 0.015, N_DAYS)
returns_dict = {}
for i, asset in enumerate(assets):
    beta = 0.5 + 0.5 * i / n_assets   # BTC has lowest beta, XRP highest
    alpha = 0.0005 * (n_assets - i) / n_assets  # BTC has highest alpha
    idio = np.random.normal(0, 0.01, N_DAYS)
    returns_dict[asset] = alpha + beta * market_factor + idio

returns_df = pd.DataFrame(returns_dict, index=dates)

# Build price panels
prices = pd.DataFrame(index=dates)
base_prices = {"BTCUSDT": 60000, "ETHUSDT": 3500, "SOLUSDT": 150, "BNBUSDT": 600, "XRPUSDT": 0.60}
for asset in assets:
    prices[asset] = base_prices[asset] * np.exp(np.cumsum(returns_df[asset]))

print(f"价格面板: {prices.shape[0]} 天 × {prices.shape[1]} 资产")
print(f"日期范围: {dates[0].date()} 至 {dates[-1].date()}")
prices.head(3)

---
## 3 · 从零构建绩效指标

In [ ]:
def annualized_sharpe(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """年化夏普比率（假设无风险利率为零）。"""
    r = returns.dropna().astype(float)
    if len(r) < 2:
        return None
    vol = float(r.std(ddof=0))
    if vol == 0:
        return None
    return float((r.mean() / vol) * math.sqrt(periods_per_year))

def annualized_sortino(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """索提诺比率 — 仅惩罚下行波动。"""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    downside = r[r < 0]
    dd = float(downside.std(ddof=0)) if not downside.empty else 0.0
    if dd == 0:
        return None
    return float((r.mean() / dd) * math.sqrt(periods_per_year))

def max_drawdown(returns: pd.Series) -> float | None:
    """从资金曲线计算最大回撤。"""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    equity = (1 + r).cumprod()
    peak = equity.cummax()
    dd = (equity / peak) - 1
    return float(dd.min())

def calmar_ratio(returns: pd.Series, periods_per_year: int = 365) -> float | None:
    """卡尔马比率 = 年化收益 / |最大回撤|。"""
    r = returns.dropna().astype(float)
    mdd = max_drawdown(r)
    if mdd is None or mdd == 0:
        return None
    ann_return = float(r.mean() * periods_per_year)
    return ann_return / abs(mdd)

def profit_factor(returns: pd.Series) -> float | None:
    """盈利因子 = 总盈利 / 总亏损。"""
    r = returns.dropna().astype(float)
    gross_profit = float(r[r > 0].sum())
    gross_loss = abs(float(r[r < 0].sum()))
    if gross_loss == 0:
        return None
    return gross_profit / gross_loss

def win_rate(returns: pd.Series) -> float | None:
    """正收益周期的占比。"""
    r = returns.dropna().astype(float)
    if r.empty:
        return None
    return float((r > 0).mean())

print("指标函数定义完成 ✅")

---
## 4 · 简单滚动前推回测器

In [ ]:
def simple_momentum_backtest(
    prices: pd.DataFrame,
    lookback: int = 5,
    top_n: int = 3,
) -> pd.Series:
    """
    简单动量策略的滚动前推回测：
    - 每天按 lookback 天收益率排名
    - 等权做多排名前 top_n 的资产
    - 持仓 1 天后再平衡
    """
    daily_returns = prices.pct_change()
    lookback_returns = prices.pct_change(lookback)
    
    strategy_returns = []
    strategy_dates = []
    
    for i in range(lookback, len(prices) - 1):
        # Rank by lookback return
        scores = lookback_returns.iloc[i].dropna()
        if len(scores) < top_n:
            continue
        top_assets = scores.nlargest(top_n).index.tolist()
        
        # Equal-weight next-day return
        next_day_ret = daily_returns.iloc[i + 1][top_assets].mean()
        strategy_returns.append(next_day_ret)
        strategy_dates.append(prices.index[i])
    
    return pd.Series(strategy_returns, index=strategy_dates, name="momentum")

# Run backtest
strat_returns = simple_momentum_backtest(prices, lookback=5, top_n=3)

# Benchmark: equal-weight buy-and-hold
benchmark_returns = prices.pct_change().mean(axis=1).iloc[5:]
benchmark_returns = benchmark_returns.reindex(strat_returns.index)

print(f"策略共 {len(strat_returns)} 个日收益")
print(f"起始: {strat_returns.index[0].date()}  结束: {strat_returns.index[-1].date()}")

---
## 5 · 训练 / 验证拆分

In [ ]:
split_point = len(strat_returns) // 2
train_returns = strat_returns.iloc[:split_point]
val_returns = strat_returns.iloc[split_point:]

train_bench = benchmark_returns.iloc[:split_point]
val_bench = benchmark_returns.iloc[split_point:]

def report_metrics(returns: pd.Series, label: str) -> dict:
    """计算并打印所有指标。"""
    metrics = {
        "夏普比率": annualized_sharpe(returns),
        "索提诺比率": annualized_sortino(returns),
        "卡尔马比率": calmar_ratio(returns),
        "最大回撤": max_drawdown(returns),
        "盈利因子": profit_factor(returns),
        "胜率": win_rate(returns),
        "日均收益": float(returns.mean()) if not returns.empty else 0,
    }
    print(f"\n{'='*40}")
    print(f" {label} ({len(returns)} 天)")
    print(f"{'='*40}")
    for name, val in metrics.items():
        if val is None:
            print(f"  {name:18s}: N/A")
        elif "率" in name or "回撤" in name:
            print(f"  {name:18s}: {val:.2%}")
        else:
            print(f"  {name:18s}: {val:.4f}")
    return metrics

train_metrics = report_metrics(train_returns, "训练集 — 动量")
val_metrics = report_metrics(val_returns, "验证集 — 动量")
bench_metrics = report_metrics(benchmark_returns, "基准 — 等权持有")

---
## 6 · 资金曲线与回撤可视化

In [ ]:
strat_equity = (1 + strat_returns).cumprod()
bench_equity = (1 + benchmark_returns.fillna(0)).cumprod()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

# Equity curves
ax1.plot(strat_equity.index, strat_equity, label="动量策略", lw=1.5, color="#3498db")
ax1.plot(bench_equity.index, bench_equity, label="等权持有基准", lw=1.5, color="#95a5a6")

# Mark train/validation split
split_date = strat_returns.index[split_point]
ax1.axvline(split_date, ls="--", color="black", lw=1, label="训练/验证分割线")
ax1.fill_between(strat_equity.index[:split_point+1], 0, strat_equity.max() * 1.1,
                 alpha=0.03, color="blue")
ax1.fill_between(strat_equity.index[split_point:], 0, strat_equity.max() * 1.1,
                 alpha=0.03, color="green")

ax1.set_ylabel("资金曲线")
ax1.set_title("滚动前推回测: 动量策略")
ax1.legend(loc="upper left")

# Drawdown
strat_peak = strat_equity.cummax()
strat_dd = (strat_equity / strat_peak) - 1
ax2.fill_between(strat_dd.index, strat_dd, alpha=0.4, color="#e74c3c")
ax2.axvline(split_date, ls="--", color="black", lw=1)
ax2.set_ylabel("回撤")
ax2.set_xlabel("日期")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 7 · 滚动夏普比率

In [ ]:
def rolling_sharpe(returns: pd.Series, window: int = 30) -> pd.Series:
    """使用固定窗口的滚动年化夏普比率。"""
    rolling_mean = returns.rolling(window).mean()
    rolling_std = returns.rolling(window).std(ddof=0)
    return (rolling_mean / rolling_std) * math.sqrt(DAYS_PER_YEAR)

rs_strat = rolling_sharpe(strat_returns, window=30)
rs_bench = rolling_sharpe(benchmark_returns, window=30)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rs_strat.index, rs_strat, label="动量策略 (30天滚动)", lw=1.5, color="#3498db")
ax.plot(rs_bench.index, rs_bench, label="基准 (30天滚动)", lw=1.5, color="#95a5a6")
ax.axhline(0, ls="-", color="black", lw=0.5)
ax.axhline(1.0, ls="--", color="green", lw=0.8, label="夏普 = 1.0")
ax.axhline(-1.0, ls="--", color="red", lw=0.8, label="夏普 = -1.0")
ax.axvline(split_date, ls="--", color="black", lw=1)
ax.set_ylabel("滚动夏普")
ax.set_title("30天滚动夏普比率")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 8 · 综合评分（生产评分体系）

机器人使用**综合评分**为策略表现排名：

$$
\text{Score} = 0.40 \times \text{Sortino} + 0.30 \times \text{Sharpe} + 0.30 \times \text{Calmar}
$$

In [ ]:
def composite_score(returns: pd.Series) -> float:
    """加权综合评分：索提诺 (40%) + 夏普 (30%) + 卡尔马 (30%)。"""
    sortino = annualized_sortino(returns) or 0.0
    sharpe = annualized_sharpe(returns) or 0.0
    calmar = calmar_ratio(returns) or 0.0
    return 0.40 * sortino + 0.30 * sharpe + 0.30 * calmar

print(f"综合评分（训练集）:  {composite_score(train_returns):.4f}")
print(f"综合评分（验证集）:  {composite_score(val_returns):.4f}")
print(f"综合评分（基准）:    {composite_score(benchmark_returns):.4f}")

---
## 9 · 参数敏感性扫描

In [ ]:
# Sweep lookback and top_n on TRAIN period only
results = []
for lookback in [3, 5, 7, 10, 14]:
    for top_n in [1, 2, 3, 4]:
        r = simple_momentum_backtest(prices, lookback=lookback, top_n=top_n)
        train_r = r.iloc[:len(r)//2]
        val_r = r.iloc[len(r)//2:]
        results.append({
            "lookback": lookback,
            "top_n": top_n,
            "train_sharpe": annualized_sharpe(train_r) or 0.0,
            "val_sharpe": annualized_sharpe(val_r) or 0.0,
            "train_score": composite_score(train_r),
            "val_score": composite_score(val_r),
        })

sweep_df = pd.DataFrame(results)

# Heatmap: train Sharpe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (ax1, "train_sharpe", "训练集夏普"),
    (ax2, "val_sharpe", "验证集夏普"),
]:
    pivot = sweep_df.pivot(index="lookback", columns="top_n", values=col)
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto",
                   vmin=-2, vmax=4)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("top_n")
    ax.set_ylabel("lookback")
    ax.set_title(title)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i,j]:.1f}", ha="center", va="center", fontsize=9)

fig.suptitle("参数敏感性: 动量回测", fontsize=13)
plt.tight_layout()
plt.show()

# Best parameters (by train composite score)
best = sweep_df.loc[sweep_df["train_score"].idxmax()]
print(f"\n最优参数（按训练集评分）: lookback={int(best['lookback'])}, top_n={int(best['top_n'])}")
print(f"  训练集评分: {best['train_score']:.4f}  |  验证集评分: {best['val_score']:.4f}")

---
## 10 · 与生产指标对比

In [ ]:
# Import production metric functions
from bot.backtest.core_module_backtester import (
    _annualized_sharpe as prod_sharpe,
    _annualized_sortino as prod_sortino,
    _max_drawdown as prod_max_dd,
    _profit_factor as prod_pf,
)

test_returns = strat_returns

comparisons = [
    ("夏普比率", annualized_sharpe(test_returns), prod_sharpe(test_returns, periods_per_year=365)),
    ("索提诺比率", annualized_sortino(test_returns), prod_sortino(test_returns, periods_per_year=365)),
    ("最大回撤", max_drawdown(test_returns), prod_max_dd(test_returns)),
    ("盈利因子", profit_factor(test_returns), prod_pf(test_returns)),
]

print(f"{'指标':18s} {'我们的':>10s} {'生产级':>12s} {'匹配':>6s}")
print("-" * 50)
for name, ours, prod in comparisons:
    if ours is None or prod is None:
        match = ours is None and prod is None
        print(f"{name:18s} {'N/A':>10s} {'N/A':>12s} {'✅' if match else '❌':>6s}")
    else:
        match = abs(ours - prod) < 1e-6
        print(f"{name:18s} {ours:10.6f} {prod:12.6f} {'✅' if match else '❌':>6s}")

---
## 11 · 收益分布分析

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(strat_returns.dropna(), bins=40, alpha=0.7, color="#3498db", label="策略", density=True)
ax1.hist(benchmark_returns.dropna(), bins=40, alpha=0.5, color="#95a5a6", label="基准", density=True)
ax1.axvline(0, ls="-", color="black", lw=0.5)
ax1.axvline(strat_returns.mean(), ls="--", color="#3498db", lw=1.5)
ax1.set_xlabel("日收益率")
ax1.set_ylabel("密度")
ax1.set_title("收益分布")
ax1.legend()

# QQ-style: cumulative returns
sorted_strat = np.sort(strat_returns.dropna())
sorted_bench = np.sort(benchmark_returns.dropna())
min_len = min(len(sorted_strat), len(sorted_bench))
ax2.scatter(sorted_bench[:min_len], sorted_strat[:min_len], s=10, alpha=0.5, color="#3498db")
ax2.plot([-0.05, 0.05], [-0.05, 0.05], ls="--", color="red", lw=1)
ax2.set_xlabel("基准分位数")
ax2.set_ylabel("策略分位数")
ax2.set_title("QQ图: 策略 vs 基准")

plt.tight_layout()
plt.show()

---
## 12 · 核心要点

| 概念 | 说明 |
|------|------|
| **滚动前推** | 前半段训练、后半段验证——避免前瞻偏差 |
| **夏普比率** | 风险调整收益；> 1.0 良好，> 2.0 优秀 |
| **索提诺比率** | 类似夏普但只惩罚下行（更适合交易策略） |
| **卡尔马比率** | CAGR ÷ 最大回撤——衡量单位最大痛苦的收益 |
| **盈利因子** | > 1.0 表示总盈利大于总亏损 |
| **综合评分** | 40% 索提诺 + 30% 夏普 + 30% 卡尔马 |
| **参数扫描** | 始终检查训练表现是否延续到验证期 |

### 回测常见陷阱

| 陷阱 | 应对方法 |
|------|----------|
| 前瞻偏差 | 仅使用决策时刻可用的数据 |
| 生存偏差 | 包含已退市资产 |
| 过拟合 | 保持参数空间小；在样本外验证 |
| 交易成本 | 将滑点和手续费计入收益 |
| 数据窥探 | 运行回测前预先注册假设 |

---
## 🔬 练习

1. **均值回归回测：** 复制第4节但每天买入表现*最差*的 N 个资产。对比动量策略的夏普比率。

2. **滚动前推优化：** 实现一个滚动60天训练窗口，每30天重新优化 `lookback` 和 `top_n`。这能否提升验证集夏普？

3. **交易成本模型：** 加入 10bp（0.1%）的往返交易成本。盈利因子受到多大影响？

4. **制度条件回测：** 按制度（来自 NB06）拆分资金曲线，分别计算牛市、震荡和熊市期间的夏普比率。

---
## ✅ 知识检查

1. 为什么索提诺比率比夏普比率更适合评估交易策略？
2. 盈利因子恰好等于 1.0 意味着什么？
3. 为什么训练集夏普 = 3.0 但验证集夏普 = 0.5？这说明了什么？
4. 综合评分如何加权各指标，为什么这样设计？
5. 列举三种回测中的前瞻偏差来源。

---
## 🔗 下一步

**[NB10 — 完整管线集成 →](10_完整管线集成.ipynb)**

我们将把所有模块串联起来：数据 → 信号 → 制度 → 集成 → 组合 → 风控 → 执行，复现生产级 `main.py` 交易循环。